# Stage 2 — Subtask 1: Dataset Selection & Taxonomy
**Leaf-JEPA IRP** | Dataset Preparation

This notebook:
1. Documents and justifies dataset selection with explicit limitation acknowledgement
2. Defines the Leaf-JEPA disease taxonomy using EPPO/USDA nomenclature
3. Exports `taxonomy.json` and `dataset_registry.json` consumed by all later notebooks

> **Why this comes first:** Every downstream decision — splits, augmentation, evaluation — depends on knowing which datasets are used and why. Documenting limitations *before* experiments prevents retroactive rationalisation.


In [ ]:
%pip install pandas --quiet

In [14]:
import sys, json
from pathlib import Path

# ── Project paths ─────────────────────────────────────────────────────────────
PROJECT_ROOT = Path("..").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / "stage2_dataset_preparation"))

OUTPUTS_DIR = PROJECT_ROOT / "stage2_dataset_preparation/outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Outputs dir  : {OUTPUTS_DIR}")


Project root : D:\IRP\leaf-jepa
Outputs dir  : D:\IRP\leaf-jepa\stage2_dataset_preparation\outputs


## 1. Dataset Registry

### Design Decision: Why These Two Datasets?

| Dataset | Role | Justification |
|---------|------|---------------|
| PlantVillage | Primary training | Largest annotated plant disease dataset (~54K images, 38 classes). Scale sufficient for SSL pretraining. |
| PlantDoc | Evaluation only | Field-condition images. Represents real deployment scenario. Tests cross-domain generalisation (RQ4). |

> **Critical restriction:** PlantDoc is **never** used for training.


In [29]:
DATASET_REGISTRY = {
    "PlantVillage": {
        "citation": (
            "Hughes, D. P., & Salathé, M. (2015). An open access repository of images "
            "on plant health to enable the development of mobile disease diagnostics. arXiv:1511.08060."
        ),
        "paper_link": (
            "https://arxiv.org/abs/1911.10317"
        ),
        "scale": {"images": 54305, "classes": 38, "crop_species": 14},
        "imaging_conditions": "Controlled lab: uniform backgrounds, consistent lighting, single leaf per image.",
        "role_in_project": "PRIMARY TRAINING DATA — SSL pretraining + supervised fine-tuning (PEFT experiments).",
        "justification": (
            "PlantVillage is the largest publicly available annotated plant disease dataset, "
            "providing the scale necessary for ViT-based SSL pretraining. Its 38-class coverage "
            "across 14 crop species defines a broad disease taxonomy for a foundational model."
        ),
        "known_limitations": {
            "lab_bias": {
                "description": "All images captured under controlled laboratory conditions.",
                "implication_for_research": (
                    "Directly motivates RQ4: cross-domain transfer to PlantDoc. "
                    "Model may learn background artefacts rather than disease features."
                ),
            },
            "class_imbalance": {
                "description": "Class sizes range ~200–5,000 images.",
                "implication_for_research": (
                    "Macro-F1 is the primary metric (not accuracy). All splits are stratified."
                ),
            },
            "near_duplicates": {
                "description": "Near-duplicate images from in-dataset augmentation during construction.",
                "implication_for_research": (
                    "pHash deduplication applied across train/test splits (Subtask 7) "
                    "to prevent test accuracy inflation."
                ),
            },
            "single_leaf_framing": {
                "description": "Single isolated leaf per image — no multi-leaf scenes.",
                "implication_for_research": (
                    "Model may not generalise to multi-leaf field images. "
                    "Acknowledged as a project limitation in the dissertation."
                ),
            },
        },
    },
    "PlantDoc": {
        "citation": (
            "Singh, D., Jain, N., Jain, P., Kayal, P., Kumawat, S., & Batra, N. (2020). "
            "PlantDoc: A Dataset for Visual Plant Disease Detection. In Proceedings of the 7th ACM IKDD CoDS and 25th COMAD (pp. 249-253)."
        ),
        "paper_link": (
            "https://arxiv.org/abs/1511.08060"
        ),
        "scale": {"images": 2578, "classes": 27, "crop_species": 13},
        "imaging_conditions": "Real-world field: variable lighting, backgrounds, angles, quality.",
        "role_in_project": "CROSS-DOMAIN EVALUATION ONLY — never used for pretraining or fine-tuning.",
        "justification": (
            "The only large-scale publicly available plant disease dataset with genuine "
            "field conditions. Evaluation-only status ensures any accuracy improvement "
            "over generic I-JEPA reflects true generalisation, not in-distribution performance."
        ),
        "known_limitations": {
            "small_scale": {
                "description": "~2,569 images — insufficient for training.",
                "implication_for_research": "Cannot be used for training. Enforced programmatically in all data loaders.",
            },
            "label_noise": {
                "description": "Crowdsourced annotations with estimated 10–15% label noise.",
                "implication_for_research": (
                    "PlantDoc accuracy is a noisy lower bound. "
                    "Small differences (<3%) on PlantDoc should not be over-interpreted."
                ),
            },
            "incomplete_overlap": {
                "description": "Only ~17–20 disease-crop combinations overlap with PlantVillage.",
                "implication_for_research": (
                    "Cross-domain evaluation uses the intersection only. "
                    "PlantDoc-exclusive classes constitute a zero-shot sub-experiment."
                ),
            },
        },
    },
}

# Save registry
registry_path = OUTPUTS_DIR / "dataset_registry.json"
with open(registry_path, "w") as f:
    json.dump({"datasets": DATASET_REGISTRY}, f, indent=2, default=str)
print(f"Dataset registry saved → {registry_path}")

# Print summary
for name, info in DATASET_REGISTRY.items():
    s = info["scale"]
    print(f"\n{'='*55}")
    print(f"{name}: {s['images']:,} images | {s['classes']} classes | {s['crop_species']} crops")
    print(f"Imaging Condition: {info['imaging_conditions'][:57]}...")
    print(f"Role: {info['role_in_project'][:65]}...")
    print(f"Limitations: {len(info['known_limitations'])} documented")


Dataset registry saved → D:\IRP\leaf-jepa\stage2_dataset_preparation\outputs\dataset_registry.json

PlantVillage: 54,305 images | 38 classes | 14 crops
Imaging Condition: Controlled lab: uniform backgrounds, consistent lighting,...
Role: PRIMARY TRAINING DATA — SSL pretraining + supervised fine-tuning ...
Limitations: 4 documented

PlantDoc: 2,578 images | 27 classes | 13 crops
Imaging Condition: Real-world field: variable lighting, backgrounds, angles,...
Role: CROSS-DOMAIN EVALUATION ONLY — never used for pretraining or fine...
Limitations: 3 documented


## 2. Disease Taxonomy

The Leaf-JEPA taxonomy uses:
- **EPPO(European and Mediterranean Plant Protection Organization) Global Database** for pathogen names (internationally comparable, used by FAO, https://gd.eppo.int)
- **USDA PLANTS Database** for crop species names
- Five pathogen categories: Fungal, Bacterial, Viral, Abiotic, Healthy

> **Why EPPO/USDA ontology instead of raw PlantVillage names?**
> PlantVillage uses informal naming like `Tomato___Early_blight`. EPPO names are used by FAO phytosanitary records and published agronomic literature, making our taxonomy cross-referenceable with official agricultural standards.

> **Why include Healthy as a class?**
> A deployment model must distinguish healthy from diseased tissue. Excluding Healthy creates a model incapable of negative predictions — a fundamental practical failure. All evaluation reports accuracy *including* the Healthy class.

Each entry includes:
  - plantvillage_label : Original class name in the PlantVillage dataset
  - eppo_pathogen      : EPPO-standard pathogen name (where applicable)
  - common_name        : Human-readable disease name
  - crop               : Affected crop species (USDA common name)
  -  pathogen_category  : One of PATHOGEN_CATEGORIES in config.py
  - plantdoc_label     : Corresponding PlantDoc class name (None if no overlap)
  - visual_symptoms    : Primary visual cues relevant to the model
  - severity           : Qualitative impact on crop yield (Low/Medium/High/Very High)


In [36]:
PATHOGEN_CATEGORIES = ["Fungal", "Bacterial", "Viral", "Abiotic", "Healthy"]

# Full taxonomy — EPPO/USDA nomenclature
# Fields: plantvillage_label, eppo_pathogen, common_name, crop,
#         pathogen_category, plantdoc_label, visual_symptoms, severity
DISEASE_TAXONOMY = [
    # ── FUNGAL ──────────────────────────────────────────────────────────────
    {"plantvillage_label": "Apple___Apple_scab",
     "eppo_pathogen": "Venturia inaequalis", "common_name": "Apple Scab",
     "crop": "Apple (Malus domestica)", "pathogen_category": "Fungal",
     "plantdoc_label": "Apple Scab Leaf",
     "visual_symptoms": "Olive-green to dark brown velvety lesions; yellowing margins; premature defoliation",
     "severity": "High"},
    {"plantvillage_label": "Apple___Black_rot",
     "eppo_pathogen": "Botryosphaeria obtusa", "common_name": "Apple Black Rot",
     "crop": "Apple (Malus domestica)", "pathogen_category": "Fungal",
     "plantdoc_label": None,
     "visual_symptoms": "Purple-bordered circular lesions; concentric ring frog-eye pattern",
     "severity": "High"},
    {"plantvillage_label": "Apple___Cedar_apple_rust",
     "eppo_pathogen": "Gymnosporangium juniperi-virginianae", "common_name": "Cedar Apple Rust",
     "crop": "Apple (Malus domestica)", "pathogen_category": "Fungal",
     "plantdoc_label": None,
     "visual_symptoms": "Bright orange-yellow spots on upper surface; tube-like structures on underside",
     "severity": "Medium"},
    {"plantvillage_label": "Cherry_(including_sour)___Powdery_mildew",
     "eppo_pathogen": "Podosphaera clandestina", "common_name": "Cherry Powdery Mildew",
     "crop": "Cherry (Prunus avium / Prunus cerasus)", "pathogen_category": "Fungal",
     "plantdoc_label": None,
     "visual_symptoms": "White powdery coating on young leaves; leaf curling; purplish discolouration",
     "severity": "Medium"},
    {"plantvillage_label": "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
     "eppo_pathogen": "Cercospora zeae-maydis", "common_name": "Gray Leaf Spot (Corn)",
     "crop": "Corn (Zea mays)", "pathogen_category": "Fungal",
     "plantdoc_label": "Corn Gray leaf spot",
     "visual_symptoms": "Rectangular grey-tan lesions bounded by leaf veins; parallel edges",
     "severity": "High"},
    {"plantvillage_label": "Corn_(maize)___Common_rust_",
     "eppo_pathogen": "Puccinia sorghi", "common_name": "Common Rust (Corn)",
     "crop": "Corn (Zea mays)", "pathogen_category": "Fungal",
     "plantdoc_label": "Corn rust leaf",
     "visual_symptoms": "Brick-red to brown oval pustules (uredia) scattered on both leaf surfaces",
     "severity": "Medium"},
    {"plantvillage_label": "Corn_(maize)___Northern_Leaf_Blight",
     "eppo_pathogen": "Setosphaeria turcica", "common_name": "Northern Leaf Blight (Corn)",
     "crop": "Corn (Zea mays)", "pathogen_category": "Fungal",
     "plantdoc_label": "Corn leaf blight",
     "visual_symptoms": "Large cigar-shaped grey-green to tan lesions; 2.5–15 cm long",
     "severity": "High"},
    {"plantvillage_label": "Grape___Black_rot",
     "eppo_pathogen": "Guignardia bidwellii", "common_name": "Grape Black Rot",
     "crop": "Grape (Vitis vinifera)", "pathogen_category": "Fungal",
     "plantdoc_label": "grape leaf black rot",
     "visual_symptoms": "Circular tan lesions with dark border; black pycnidia in lesion centres",
     "severity": "Very High"},
    {"plantvillage_label": "Potato___Early_blight",
     "eppo_pathogen": "Alternaria solani", "common_name": "Potato Early Blight",
     "crop": "Potato (Solanum tuberosum)", "pathogen_category": "Fungal",
     "plantdoc_label": "Potato leaf early blight",
     "visual_symptoms": "Dark brown concentric-ring lesions (target-board); yellow halo; older leaves first",
     "severity": "Medium"},
    {"plantvillage_label": "Potato___Late_blight",
     "eppo_pathogen": "Phytophthora infestans", "common_name": "Potato Late Blight",
     "crop": "Potato (Solanum tuberosum)", "pathogen_category": "Fungal",
     "plantdoc_label": "Potato leaf late blight",
     "visual_symptoms": "Water-soaked dark lesions; white mould underside in humid conditions; rapid progression",
     "severity": "Very High"},
    {"plantvillage_label": "Tomato___Early_blight",
     "eppo_pathogen": "Alternaria solani", "common_name": "Tomato Early Blight",
     "crop": "Tomato (Solanum lycopersicum)", "pathogen_category": "Fungal",
     "plantdoc_label": "Tomato Early blight leaf",
     "visual_symptoms": "Dark concentric-ring lesions; yellow halo; lower/older leaves first",
     "severity": "Medium"},
    {"plantvillage_label": "Tomato___Late_blight",
     "eppo_pathogen": "Phytophthora infestans", "common_name": "Tomato Late Blight",
     "crop": "Tomato (Solanum lycopersicum)", "pathogen_category": "Fungal",
     "plantdoc_label": "Tomato leaf late blight",
     "visual_symptoms": "Greasy grey-green lesions; white sporulation underside; rapid browning",
     "severity": "Very High"},
    {"plantvillage_label": "Tomato___Leaf_Mold",
     "eppo_pathogen": "Passalora fulva", "common_name": "Tomato Leaf Mold",
     "crop": "Tomato (Solanum lycopersicum)", "pathogen_category": "Fungal",
     "plantdoc_label": None,
     "visual_symptoms": "Pale green-yellow patches upper surface; olive-brown velvety mould underside",
     "severity": "Medium"},
    {"plantvillage_label": "Tomato___Septoria_leaf_spot",
     "eppo_pathogen": "Septoria lycopersici", "common_name": "Tomato Septoria Leaf Spot",
     "crop": "Tomato (Solanum lycopersicum)", "pathogen_category": "Fungal",
     "plantdoc_label": "Tomato Septoria leaf spot",
     "visual_symptoms": "Small circular spots; dark border; grey-white centre with black pycnidia",
     "severity": "High"},
    {"plantvillage_label": "Tomato___Target_Spot",
     "eppo_pathogen": "Corynespora cassiicola", "common_name": "Tomato Target Spot",
     "crop": "Tomato (Solanum lycopersicum)", "pathogen_category": "Fungal",
     "plantdoc_label": None,
     "visual_symptoms": "Concentric ring lesions; dark brown with lighter centre",
     "severity": "Medium"},
    {"plantvillage_label": "Strawberry___Leaf_scorch",
     "eppo_pathogen": "Diplocarpon earlianum", "common_name": "Strawberry Leaf Scorch",
     "crop": "Strawberry (Fragaria × ananassa)", "pathogen_category": "Fungal",
     "plantdoc_label": None,
     "visual_symptoms": "Irregular dark purple spots; necrotic grey centres; marginal leaf burn",
     "severity": "Medium"},
    {"plantvillage_label": "Squash___Powdery_mildew",
     "eppo_pathogen": "Podosphaeria xanthii", "common_name": "Squash Powdery Mildew",
     "crop": "Squash (Cucurbita pepo)", "pathogen_category": "Fungal",
     "plantdoc_label": None,
     "visual_symptoms": "White powdery circular patches on upper surface; lesions coalesce",
     "severity": "Medium"},
    {"plantvillage_label": "Grape___Esca_(Black_Measles)",
     "eppo_pathogen": "Phaeomoniella chlamydospora / Phaeoacremonium spp.",
     "common_name": "Grape Esca (Black Measles)",
     "crop": "Grape (Vitis vinifera)", "pathogen_category": "Fungal",
     "plantdoc_label": None,
     "visual_symptoms": "Interveinal chlorosis and necrosis; tiger-stripe pattern; marginal scorch",
     "severity": "Very High"},
    {"plantvillage_label": "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)",
     "eppo_pathogen": "Pseudocercospora vitis", "common_name": "Grape Leaf Spot",
     "crop": "Grape (Vitis vinifera)", "pathogen_category": "Fungal",
     "plantdoc_label": None,
     "visual_symptoms": "Irregular dark brown spots; yellowish halo; premature defoliation",
     "severity": "Medium"},
    # ── BACTERIAL ───────────────────────────────────────────────────────────
    {"plantvillage_label": "Orange___Haunglongbing_(Citrus_greening)",
     "eppo_pathogen": "Candidatus Liberibacter spp.", "common_name": "Citrus Greening (HLB)",
     "crop": "Orange (Citrus sinensis)", "pathogen_category": "Bacterial",
     "plantdoc_label": None,
     "visual_symptoms": "Asymmetric blotchy mottling; yellowing not following vein pattern",
     "severity": "Very High"},
    {"plantvillage_label": "Peach___Bacterial_spot",
     "eppo_pathogen": "Xanthomonas arboricola pv. pruni", "common_name": "Peach Bacterial Spot",
     "crop": "Peach (Prunus persica)", "pathogen_category": "Bacterial",
     "plantdoc_label": None,
     "visual_symptoms": "Small water-soaked spots → dark angular lesions; shot-hole effect",
     "severity": "Medium"},
    {"plantvillage_label": "Pepper,_bell___Bacterial_spot",
     "eppo_pathogen": "Xanthomonas euvesicatoria", "common_name": "Bell Pepper Bacterial Spot",
     "crop": "Bell Pepper (Capsicum annuum)", "pathogen_category": "Bacterial",
     "plantdoc_label": None,
     "visual_symptoms": "Small water-soaked spots with yellow halo; greasy appearance",
     "severity": "Medium"},
    {"plantvillage_label": "Tomato___Bacterial_spot",
     "eppo_pathogen": "Xanthomonas vesicatoria", "common_name": "Tomato Bacterial Spot",
     "crop": "Tomato (Solanum lycopersicum)", "pathogen_category": "Bacterial",
     "plantdoc_label": "Tomato leaf bacterial spot",
     "visual_symptoms": "Small water-soaked spots; yellow halo; dark brown necrotic centres",
     "severity": "Medium"},
    # ── VIRAL ────────────────────────────────────────────────────────────────
    {"plantvillage_label": "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
     "eppo_pathogen": "Tomato Yellow Leaf Curl Virus (TYLCV)",
     "common_name": "Tomato YLCV",
     "crop": "Tomato (Solanum lycopersicum)", "pathogen_category": "Viral",
     "plantdoc_label": "Tomato leaf yellow virus",
     "visual_symptoms": "Upward leaf curl; marginal yellowing; small crumpled leaves; stunted growth",
     "severity": "Very High"},
    {"plantvillage_label": "Tomato___Tomato_mosaic_virus",
     "eppo_pathogen": "Tomato Mosaic Virus (ToMV)", "common_name": "Tomato Mosaic Virus",
     "crop": "Tomato (Solanum lycopersicum)", "pathogen_category": "Viral",
     "plantdoc_label": "Tomato leaf mosaic virus",
     "visual_symptoms": "Mottled light/dark green mosaic pattern; leaf distortion; fern-like",
     "severity": "High"},
    # ── ABIOTIC ──────────────────────────────────────────────────────────────
    {"plantvillage_label": "Tomato___Spider_mites Two-spotted_spider_mite",
     "eppo_pathogen": "Tetranychus urticae (arachnid pest)",
     "common_name": "Tomato Spider Mite Damage",
     "crop": "Tomato (Solanum lycopersicum)", "pathogen_category": "Abiotic",
     "plantdoc_label": None,
     "visual_symptoms": "Stippled/speckled yellowing; fine webbing underside; bronzing; desiccation",
     "severity": "Medium"},
    # ── HEALTHY ──────────────────────────────────────────────────────────────
    {"plantvillage_label": "Apple___healthy", "eppo_pathogen": None,
     "common_name": "Healthy Apple", "crop": "Apple (Malus domestica)",
     "pathogen_category": "Healthy", "plantdoc_label": "Apple leaf",
     "visual_symptoms": "Uniform green; serrated margins; no scab or rust", "severity": "None"},
    {"plantvillage_label": "Blueberry___healthy", "eppo_pathogen": None,
     "common_name": "Healthy Blueberry", "crop": "Blueberry (Vaccinium corymbosum)",
     "pathogen_category": "Healthy", "plantdoc_label": None,
     "visual_symptoms": "Uniform green; no lesions; intact ovate leaves", "severity": "None"},
    {"plantvillage_label": "Cherry_(including_sour)___healthy", "eppo_pathogen": None,
     "common_name": "Healthy Cherry", "crop": "Cherry (Prunus avium / Prunus cerasus)",
     "pathogen_category": "Healthy", "plantdoc_label": None,
     "visual_symptoms": "Uniform green; no spots; intact lanceolate leaves", "severity": "None"},
    {"plantvillage_label": "Corn_(maize)___healthy", "eppo_pathogen": None,
     "common_name": "Healthy Corn", "crop": "Corn (Zea mays)",
     "pathogen_category": "Healthy", "plantdoc_label": None,
     "visual_symptoms": "Uniform green; parallel venation; no lesions or pustules", "severity": "None"},
    {"plantvillage_label": "Grape___healthy", "eppo_pathogen": None,
     "common_name": "Healthy Grape", "crop": "Grape (Vitis vinifera)",
     "pathogen_category": "Healthy", "plantdoc_label": "grape leaf",
     "visual_symptoms": "Uniform green; palmate lobed; no mould or spots", "severity": "None"},
    {"plantvillage_label": "Pepper,_bell___healthy", "eppo_pathogen": None,
     "common_name": "Healthy Bell Pepper", "crop": "Bell Pepper (Capsicum annuum)",
     "pathogen_category": "Healthy", "plantdoc_label": None,
     "visual_symptoms": "Uniform green; ovate; no spots; intact margins", "severity": "None"},
    {"plantvillage_label": "Potato___healthy", "eppo_pathogen": None,
     "common_name": "Healthy Potato", "crop": "Potato (Solanum tuberosum)",
     "pathogen_category": "Healthy", "plantdoc_label": None,
     "visual_symptoms": "Uniform green; pinnate compound leaves; no lesions", "severity": "None"},
    {"plantvillage_label": "Raspberry___healthy", "eppo_pathogen": None,
     "common_name": "Healthy Raspberry", "crop": "Raspberry (Rubus idaeus)",
     "pathogen_category": "Healthy", "plantdoc_label": None,
     "visual_symptoms": "Uniform dark green; no lesions; intact serrated margins", "severity": "None"},
    {"plantvillage_label": "Soybean___healthy", "eppo_pathogen": None,
     "common_name": "Healthy Soybean", "crop": "Soybean (Glycine max)",
     "pathogen_category": "Healthy", "plantdoc_label": None,
     "visual_symptoms": "Uniform green; no spots; intact trifoliate structure", "severity": "None"},
    {"plantvillage_label": "Strawberry___healthy", "eppo_pathogen": None,
     "common_name": "Healthy Strawberry", "crop": "Strawberry (Fragaria × ananassa)",
     "pathogen_category": "Healthy", "plantdoc_label": None,
     "visual_symptoms": "Uniform green; no spots; intact trifoliate leaves", "severity": "None"},
    {"plantvillage_label": "Tomato___healthy", "eppo_pathogen": None,
     "common_name": "Healthy Tomato", "crop": "Tomato (Solanum lycopersicum)",
     "pathogen_category": "Healthy", "plantdoc_label": "Tomato leaf",
     "visual_symptoms": "Uniform dark green; pinnate compound leaves; no lesions", "severity": "None"},
]

print(f"Total taxonomy entries : {len(DISEASE_TAXONOMY)}")
for cat in PATHOGEN_CATEGORIES:
    n = sum(1 for e in DISEASE_TAXONOMY if e["pathogen_category"] == cat)
    print(f"  {cat:12s}: {n}")


Total taxonomy entries : 37
  Fungal      : 19
  Bacterial   : 4
  Viral       : 2
  Abiotic     : 1
  Healthy     : 11


## 3. Validate Taxonomy & Export

In [37]:
import pandas as pd

# ── Validation ───────────────────────"─────────────────────────────────────────
required_keys = {"plantvillage_label","eppo_pathogen","common_name","crop",
                 "pathogen_category","plantdoc_label","visual_symptoms","severity"}

errors = []
for i, entry in enumerate(DISEASE_TAXONOMY):
    missing = required_keys - set(entry.keys())
    if missing:
        errors.append(f"Entry {i}: missing keys {missing}")
    if entry["pathogen_category"] not in PATHOGEN_CATEGORIES:
        errors.append(f"Entry {i}: unknown category '{entry['pathogen_category']}'")

# Healthy class check (common pitfall: accidentally dropping healthy)
healthy = [e for e in DISEASE_TAXONOMY if e["pathogen_category"] == "Healthy"]
assert len(healthy) > 0, "CRITICAL: No Healthy class entries — model cannot make negative predictions!"
print(f"✅ Healthy class: {len(healthy)} entries across {len({e['crop'] for e in healthy})} crops")

if errors:
    for e in errors:
        print(f"❌ {e}")
else:
    print(f"✅ Taxonomy validation passed: {len(DISEASE_TAXONOMY)} entries, all keys present")

# ── Cross-dataset overlap ──────────────────────────────────────────────────────
shared = [e for e in DISEASE_TAXONOMY if e["plantdoc_label"] is not None]
pv_only = [e for e in DISEASE_TAXONOMY if e["plantdoc_label"] is None]
print(f"\nCross-dataset overlap :")
print(f"  Shared (PV ∩ PlantDoc) : {len(shared)}")
print(f"  PlantVillage only      : {len(pv_only)}")

# ── Export taxonomy JSON ──────────────────────────────────────────────────────
taxonomy_data = {
    "metadata": {
        "nomenclature": "EPPO Global Database / USDA PLANTS Database",
        "categories": PATHOGEN_CATEGORIES,
        "total_entries": len(DISEASE_TAXONOMY),
        "shared_with_plantdoc": len(shared),
    },
    "entries": DISEASE_TAXONOMY,
}
taxonomy_path = OUTPUTS_DIR / "taxonomy.json"
with open(taxonomy_path, "w") as f:
    json.dump(taxonomy_data, f, indent=2, default=str)
print(f"\n✅ Taxonomy exported → {taxonomy_path}")

# ── Export as readable CSV ────────────────────────────────────────────────────
(OUTPUTS_DIR / "taxonomy").mkdir(exist_ok=True)
df = pd.DataFrame(DISEASE_TAXONOMY).sort_values(["pathogen_category", "crop"])
df["has_plantdoc_equivalent"] = df["plantdoc_label"].notna()
df.to_csv(OUTPUTS_DIR /  "taxonomy_table.csv", index=False)
print(f"✅ Taxonomy CSV exported → outputs/taxonomy_table.csv")
df[["common_name","crop","pathogen_category","severity","has_plantdoc_equivalent"]].head(10)


✅ Healthy class: 11 entries across 11 crops
✅ Taxonomy validation passed: 37 entries, all keys present

Cross-dataset overlap :
  Shared (PV ∩ PlantDoc) : 16
  PlantVillage only      : 21

✅ Taxonomy exported → D:\IRP\leaf-jepa\stage2_dataset_preparation\outputs\taxonomy.json
✅ Taxonomy CSV exported → outputs/taxonomy_table.csv


,common_name,crop,pathogen_category,severity,has_plantdoc_equivalent
25,Tomato Spider Mite Damage,Tomato (Solanum lycopersicum),Abiotic,Medium,False
21,Bell Pepper Bacterial Spot,Bell Pepper (Capsicum annuum),Bacterial,Medium,False
19,Citrus Greening (HLB),Orange (Citrus sinensis),Bacterial,Very High,False
20,Peach Bacterial Spot,Peach (Prunus persica),Bacterial,Medium,False
22,Tomato Bacterial Spot,Tomato (Solanum lycopersicum),Bacterial,Medium,True
0,Apple Scab,Apple (Malus domestica),Fungal,High,True
1,Apple Black Rot,Apple (Malus domestica),Fungal,High,False
2,Cedar Apple Rust,Apple (Malus domestica),Fungal,Medium,False
3,Cherry Powdery Mildew,Cherry (Prunus avium / Prunus cerasus),Fungal,Medium,False
4,Gray Leaf Spot (Corn),Corn (Zea mays),Fungal,High,True
